# K-Nearest Neighbors Benchmarking

In this notebook, we compare our custom-built k-NN classifier against the `scikit-learn` implementation. We will test our custom distance metrics (Euclidean, Manhattan, Minkowski) and evaluate the algorithm's performance on the Breast Cancer dataset.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
from time import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN

# Import our custom modules
from classical_ml.neighbors.knn import KNN as CustomKNN
from utils.metrics import accuracy_score

d:\Github\Classical-ML-From-Scratch\utils\distances.py:6: SyntaxWarning: invalid escape sequence '\s'
  Formula: \sqrt{\sum_{i=1}^{n} (x1_i - x2_i)^2}
d:\Github\Classical-ML-From-Scratch\utils\distances.py:13: SyntaxWarning: invalid escape sequence '\s'
  Formula: \sum_{i=1}^{n} |x1_i - x2_i|
d:\Github\Classical-ML-From-Scratch\utils\distances.py:20: SyntaxWarning: invalid escape sequence '\s'
  Formula: (\sum_{i=1}^{n} |x1_i - x2_i|^p)^{1/p}


In [2]:
# 1. Load Dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale the features (Crucial for distance-based algorithms like k-NN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training data shape: {X_train_scaled.shape}")
print(f"Testing data shape: {X_test_scaled.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


In [3]:
print("1. Custom k-NN (Euclidian Distance)")
k_value = 5
start = time()

# Initialize and fit (Lazy learner: only stores data)
custom_knn_euc = CustomKNN(k=k_value, metric='euclidean')
custom_knn_euc.fit(X_train_scaled, y_train)

# Predict
preds_euc = custom_knn_euc.predict(X_test_scaled)
time_euc = time() - start

print(f"Accuracy   : {accuracy_score(y_test, preds_euc):.4f}")
print(f"Time Taken : {time_euc:.4f} seconds")

1. Custom k-NN (Euclidian Distance)
Accuracy   : 0.9474
Time Taken : 0.3492 seconds


In [4]:
print("2. Custom k-NN (Manhattan Distance)")
start = time()

custom_knn_man = CustomKNN(k=k_value, metric='manhattan')
custom_knn_man.fit(X_train_scaled, y_train)
preds_man = custom_knn_man.predict(X_test_scaled)
time_man = time() - start

print(f"Accuracy   : {accuracy_score(y_test, preds_man):.4f}")
print(f"Time Taken : {time_man:.4f} seconds")

2. Custom k-NN (Manhattan Distance)
Accuracy   : 0.9649
Time Taken : 0.2758 seconds


In [5]:
print("3. Scikit-Learn k-NN")
start = time()

sk_knn = SklearnKNN(n_neighbors=k_value, metric='euclidean')
sk_knn.fit(X_train_scaled, y_train)
preds_sk = sk_knn.predict(X_test_scaled)
time_sk = time() - start

print(f"Accuracy   : {accuracy_score(y_test, preds_sk):.4f}")
print(f"Time Taken : {time_sk:.4f} seconds")

3. Scikit-Learn k-NN
Accuracy   : 0.9474
Time Taken : 2.6495 seconds


## Conclusion
The custom k-NN implementation achieves the exact same accuracy as `scikit-learn` when using the Euclidean metric. This verifies the correctness of our distance calculations and majority voting logic.

However, notice the **Time Taken**. Our pure Python/NumPy implementation is noticeably slower during the `.predict()` phase compared to `scikit-learn`. This highlights a fundamental concept in k-NN: it is a **lazy learner**. While training time is near zero (O(1)), prediction time is extremely high (O(N*M) where N is number of training samples and M is number of features). `scikit-learn` overcomes this by using advanced data structures like **KD-Trees** or **Ball Trees** implemented in C to heavily optimize neighbor search, whereas our implementation uses brute-force linear scanning.